# Band edge -- Figures 9, 10 and 11

The field response near the absorption edge, on the valley disks (all three field terms, $\nu = 5$ meV). Fig. 9 is $\eta(B)$ just above the gap, Fig. 10 the field coefficient across the first 0.45 eV, and Fig. 11 compares the threshold coefficient across models.

## Load engine

In [ ]:
%run engine.ipynb

## Resolution

Paper settings, with a quick setting in the comment. The cached results in `data/` have the resolution in their file names, so changing a line here starts a fresh run for that section.

In [ ]:
R_disk = 0.9       # disk radius (1/a)
h_edge = 0.0007    # disk spacing (1/a) in Fig. 9; 0.005 for a quick look
dB_edge = 2        # field step (T) in Fig. 9, the LDA runs always use 10 T; 10 for a quick look
h_chi = 0.001      # disk spacing (1/a) in Fig. 10; 0.002 for a quick look (coarser grids give spurious sign changes)
d_off = 0.001      # photon-energy step (eV) in Fig. 10; 0.005 for a quick look
N_kp = 401         # k.p grid for the squares of Fig. 11; 301 for a quick look

## Figure 9 -- $\eta^{x;xx}(B)$ at the band edge

At $E_g + 0.01$ eV, averaged over five photon energies from $E_g + 0.0075$ to $E_g + 0.0125$ eV. The threshold coefficient $\chi$ is the slope $s$ of $\eta = sB + cB^3$ fitted to the mirrored data. Fig. 11 also needs the NN model and the LDA parameters, so they are computed here too. This is the slowest part: about 2.5 min per field at $h = 0.0007/a$ (about $10^7$ points), so about 75 min per curve, 10 h for the eight GGA curves and 2 h for the eight LDA ones.

In [ ]:
edge = {}
for func, dB in (("GGA", dB_edge), ("LDA", 10)):
    B_edge = np.arange(0, 60 + 1e-9, dB)
    for model in ("TNN", "NN"):
        for m in materials:
            fn = f"data/edge_{m}_{model}_{func}_R{R_disk}_h{h_edge}_dB{dB}.npz"
            if not os.path.exists(fn):
                Eg = gap(m, model, func)
                omega = Eg + 0.01 + np.linspace(-0.0025, 0.0025, 5)
                grids = valley_grids(R_disk, h_edge)
                eta = []
                for B in B_edge:
                    eta.append(eta_mag(m, model, B, omega, 0.005, grids, func)[:, 0, 0, 0].mean())
                np.savez(fn, B=B_edge, eta=eta_SI(params(m, func)["a"]) * np.array(eta))
            d = np.load(fn)
            B_f, eta_f = mirror(d["B"], d["eta"])
            s, c = np.linalg.lstsq(np.stack([B_f, B_f**3], axis=1), eta_f, rcond=None)[0]
            edge[m, model, func] = dict(B=d["B"], eta=d["eta"], chi=s, cubic=100 * c * 60**2 / s)

for func in ("GGA", "LDA"):
    for model in ("TNN", "NN"):
        print(f"{func} {model:3s} chi (1e-4 A m V^-2 s^-1 T^-1):", np.round([1e4 * edge[m, model, func]["chi"] for m in materials], 3),
              " cubic part at 60 T (%):", np.round([edge[m, model, func]["cubic"] for m in materials], 2))

In [ ]:
y_max = max(np.abs(edge[m, "TNN", "GGA"]["eta"]).max() for m in materials)
p10 = int(np.floor(np.log10(y_max)))
fig, ax = plt.subplots(1, 4, figsize=(TW, 1.85), gridspec_kw=dict(wspace=0.42))
for i, m in enumerate(materials):
    d = edge[m, "TNN", "GGA"]
    B_f, eta_f = mirror(d["B"], d["eta"])
    ax[i].plot(B_f, eta_f / 10.0**p10, color=RED if d["chi"] > 0 else BLUE)
    zero(ax[i], v=True)
    ax[i].set_xlabel(r"$B$ (T)")
    ax[i].set_xlim(-62, 62)
    ax[i].xaxis.set_major_locator(MaxNLocator(5, steps=[1, 2, 3, 5, 6, 10]))
    ax[i].yaxis.set_major_locator(MaxNLocator(5))
    ax[i].set_title(names[m], pad=3, loc="right")
    label(ax[i], "abcd"[i])
ax[0].set_ylabel(r"$\eta^{x;xx}$ ($10^{%d}\,%s$)" % (p10, UNIT))
save(fig, "lpge_edge_sign_grid")
plt.show()

## Figure 10 -- field coefficient versus photon energy

$\chi(\omega) = \eta^{x;xx}(B = 30\,{\rm T})/30$ T for $\hbar\omega - E_g$ from $-0.03$ to $0.45$ eV, TNN and NN, smoothed by a 5 meV running mean. Open circles mark sign changes and the dotted line the onset $E_g + 2\lambda$ of the transition from the lower spin-split valence band (B). About 2 min per curve, 15 min in all at paper settings.

In [ ]:
off = np.arange(-0.03, 0.45 + 1e-9, d_off)
chi_w = {}
for model in ("TNN", "NN"):
    for m in materials:
        fn = f"data/chi_omega_{m}_{model}_R{R_disk}_h{h_chi}_d{d_off}.npz"
        if not os.path.exists(fn):
            eta = eta_mag(m, model, 30.0, gap(m, model) + off, 0.005, valley_grids(R_disk, h_chi))[:, 0, 0, 0]
            np.savez(fn, off=off, chi=eta_SI(params(m)["a"]) * eta / 30)
        chi_w[m, model] = np.load(fn)["chi"]

In [ ]:
def crossings(x, y):
    i = np.where(np.sign(y[1:]) * np.sign(y[:-1]) < 0)[0]
    return x[i] - y[i] * (x[i + 1] - x[i]) / (y[i + 1] - y[i])


n_avg = max(1, int(round(0.005 / d_off)))
fig, ax = plt.subplots(1, 4, figsize=(TW, 2.25), gridspec_kw=dict(wspace=0.36))
for i, m in enumerate(materials):
    ys = []
    for model, c in (("TNN", RED), ("NN", ORANGE)):
        y = np.convolve(1e4 * chi_w[m, model], np.ones(n_avg) / n_avg, "same")
        k = (off >= -0.02) & (off <= 0.45)
        ax[i].plot(off[k], y[k], color=c, lw=1.0, label=model)
        x0 = crossings(off[k], y[k])
        x0 = x0[x0 > 0.004]
        ax[i].plot(x0, np.zeros_like(x0), "o", ms=4, mfc="white", mec=c, mew=1.0, zorder=5)
        print(f"{m} {model}: chi(Eg + 0.01) = {np.interp(0.01, off, y):+.3f} x 1e-4, sign changes at Eg +", np.round(x0, 3), "eV")
        ys.append(y[k])
    lo, hi = min(np.min(ys), 0), max(np.max(ys), 0)
    ax[i].set_ylim(lo - 0.08 * (hi - lo), hi + 0.08 * (hi - lo))
    ax[i].axvline(0, color="0.6", lw=0.6)
    if 2 * lambda_SOC[m] < 0.45:
        ax[i].axvline(2 * lambda_SOC[m], color="0.5", lw=0.6, ls=":")
        ax[i].text(2 * lambda_SOC[m] - 0.008, 0.93, "B", transform=ax[i].get_xaxis_transform(), fontsize=7, color="0.35", ha="right")
    zero(ax[i])
    ax[i].set_xlim(-0.02, 0.45)
    ax[i].set_xlabel(r"$\hbar\omega-E_g$ (eV)")
    ax[i].yaxis.set_major_locator(MaxNLocator(6))
    ax[i].set_title(names[m], pad=3, loc="right")
    label(ax[i], "abcd"[i])
ax[0].set_ylabel(r"$\chi$ ($10^{-4}\,%s\,\mathrm{T^{-1}}$)" % UNIT)
fig.legend(*ax[0].get_legend_handles_labels(), loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.07))
save(fig, "chi_vs_offset")
plt.show()

## Figure 11 -- threshold field coefficient across models

Tight-binding $\chi$ from the fits of Fig. 9 (GGA filled, LDA open), against the material in (a) and against $\kappa$ from the Löwdin reduction in (b), with a linear fit to the eight GGA points. The squares are the k.p model with first-principles parameters, saved by `kp.ipynb` (`kp_chi` recomputes them in a few seconds if the file is missing).

In [ ]:
kap = {}
for func in ("GGA", "LDA"):
    for model in ("TNN", "NN"):
        for m in materials:
            kap[m, model, func] = warping(m, func, model)
chi_kp = kp_chi(N_kp)
print("k.p chi (1e-4 A m V^-2 s^-1 T^-1):", np.round([1e4 * chi_kp[m] for m in materials], 3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(TW, 2.35), gridspec_kw=dict(wspace=0.28))
def marker(c, fill, lab):
    return dict(color=c, mfc=c if fill else "none", mew=0.9, ms=4.5, ls="none", label=lab)


x_gga, y_gga = [], []
for k, (model, mk, c) in enumerate((("TNN", "o", RED), ("NN", "^", ORANGE))):
    chi = [1e4 * edge[m, model, "GGA"]["chi"] for m in materials]
    kap_GGA = [kap[m, model, "GGA"] for m in materials]
    ax[0].plot(np.arange(4) + (k - 1) * 0.16, chi, mk, **marker(c, True, "TB, " + model))
    ax[1].plot(kap_GGA, chi, mk, **marker(c, True, "TB, " + model))
    x_gga += kap_GGA
    y_gga += chi
kp_label = r"$\mathbf{k}\cdot\mathbf{p}$, first-principles"
ax[0].plot(np.arange(4) + 0.16, [1e4 * chi_kp[m] for m in materials], "s", **marker(BLUE, False, kp_label))
ax[1].plot([korm[m][2] for m in materials], [1e4 * chi_kp[m] for m in materials], "s", **marker(BLUE, False, kp_label))
for model, mk, c in (("TNN", "o", RED), ("NN", "^", ORANGE)):
    ax[1].plot([kap[m, model, "LDA"] for m in materials], [1e4 * edge[m, model, "LDA"]["chi"] for m in materials], mk,
               **marker(c, False, f"TB, {model} (LDA)"))

slope, intercept = np.polyfit(x_gga, y_gga, 1)
r = np.corrcoef(x_gga, y_gga)[0, 1]
xs = np.linspace(-1.6, 0.7, 50)
ax[1].plot(xs, slope * xs + intercept, ":", color="0.4", lw=0.9, label="fit to TB (GGA)")
print(f"Linear fit to the 8 GGA TB points: chi = ({slope:.3f} kappa {intercept:+.3f}) x 1e-4, r = {r:.3f}")

ax[0].set_xticks(range(4))
ax[0].set_xticklabels([names[m] for m in materials])
ax[0].tick_params(axis="x", which="minor", bottom=False, top=False)
ax[0].set_xlim(-0.5, 3.5)
ya = y_gga + [1e4 * chi_kp[m] for m in materials]
lo, hi = min(min(ya), 0), max(max(ya), 0)
ax[0].set_ylim(lo - 0.06 * (hi - lo), hi + 0.45 * (hi - lo))
ax[0].set_ylabel(r"$\chi(E_g+0.01\,\mathrm{eV})$ ($10^{-4}\,%s\,\mathrm{T^{-1}}$)" % UNIT)
ax[1].set_xlabel(r"$\kappa$ (eV$\,\mathrm{\AA}^2$)")
for i, axis in enumerate(ax):
    zero(axis, v=(i == 1))
    label(axis, "ab"[i])
ax[0].legend(loc="upper left")
ax[1].legend(loc="upper right")
save(fig, "threshold_models")
plt.show()